In [1]:
import os
import requests

In [2]:
# Download the training text to local file
if not os.path.exists("the-verdict.txt"):
    url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"
    file_path = "the-verdict.txt"
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with open(file_path, "wb") as f:
        f.write(response.content)

In [ ]:
# Read the training text
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
print(f"Total number of characters: {len(raw_text)}")
print(raw_text[:80])

Total number of character: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow en


In [5]:
# Cleat test: use regex to split on punctuation and white-space
import re
text = "Hello, world. This, is a test."
result = re.split(r'([,.:;?_!\"()\']|--|\s)', text)
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']


In [6]:
# Clean training text
preprocessed = re.split(r'([,.:;?_!\"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(f"Total number of words: {len(preprocessed)}")
print(preprocessed[:30])

Total number of words: 4690
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


In [7]:
# Compile sorted unique word list
all_tokens = sorted(list(set(preprocessed)))

# Add special tokens for end-of-text, unknown word
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab_size = len(all_tokens)
print(f"Total number of unique tokens: {vocab_size}")

Total number of unique tokens: 1132


In [8]:
# Create a vocabulary dictionary
vocab = { token:integer for integer,token in enumerate(all_tokens) }
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 30:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)


In [9]:
# Define vocabulary encoder/decoder
class SimpleTokenizerV2:
    def __init__(self, vocab):
        # Setup word to numeric dictionary
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items() }

    def encode(self, text):
        # Encode text into numeric vector
        preprocessed = re.split(r'([,.:;?_!\"()\']|--|\s)', text)
        preprocessed = [ item.strip() for item in preprocessed if item.strip() ]

        # Handle unknown words
        preprocessed = [ item if item in self.str_to_int else "<|unk|>" for item in preprocessed ]

        # Convert words to numeric vector
        ids = [ self.str_to_int[s] for s in preprocessed ]
        return ids

    def decode(self, ids):
        # Decode numeric vector into text
        text = " ".join([self.int_to_str[i] for i in ids])
        
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!\"()\'])', r'\1', text)
        return text

In [10]:
# Test vocabulary encoder
tokenizer = SimpleTokenizerV2(vocab)
text = "\"It's the last he painted, you know,\" Mrs. Gisburn said with pardonable pride."
encoded_ids = tokenizer.encode(text)
print(encoded_ids)

# Test vocabulary decoder
decoded_text = tokenizer.decode(tokenizer.encode(text))
print(decoded_text)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]
" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


In [14]:
# Test vocabulary encoder
tokenizer = SimpleTokenizerV2(vocab)
text = "Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace."
encoded_ids = tokenizer.encode(text)
print(encoded_ids)

# Test vocabulary decoder
decoded_text = tokenizer.decode(tokenizer.encode(text))
print(decoded_text)

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]
<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


In [13]:
# Set standard tokenizer
from importlib.metadata import version
import tiktoken
print(f"tiktoken version: {version('tiktoken')}")

tiktoken version: 0.12.0


In [16]:
# Test GPT-2 encoder
tokenizer = tiktoken.get_encoding("gpt2")
text = "Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace."
encoded_ids = tokenizer.encode(text, allowed_special={ "<|endoftext|>" })
print(encoded_ids)

# Test GPT-2 decoder
decoded_text = tokenizer.decode(tokenizer.encode(text, allowed_special={ "<|endoftext|>" }))
print(decoded_text)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 262, 20562, 13]
Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [17]:
# Read the training text
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
print(f"Total number of characters: {len(raw_text)}")

Total number of characters: 20479


In [18]:
# Encode training text
enc_text = tokenizer.encode(raw_text)
print(f"Total number of characters: {len(enc_text)}")

Total number of characters: 5145


In [19]:
# Create text chunk
enc_sample = enc_text[50:]
context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]
print(f"x: {x}\ny: {y}")

x: [290, 4920, 2241, 287]
y: [4920, 2241, 287, 257]


In [ ]:
# Show encoded prediction
for i in range(1, context_size+1):
  context = enc_sample[:i]
  desired = enc_sample[i]
  print(context, "->", desired)

[290] -> 4920
[290, 4920] -> 2241
[290, 4920, 2241] -> 287
[290, 4920, 2241, 287] -> 257


In [ ]:
# Show decoded prediction
for i in range(1, context_size+1):
  context = enc_sample[:i]
  desired = enc_sample[i]
  print(tokenizer.decode(context), "->", tokenizer.decode([desired]))

 and ->  established
 and established ->  himself
 and established himself ->  in
 and established himself in ->  a


In [ ]:
# Install PyTorch
import torch
print(f"PyTorch version: {torch.__version__}")

from torch.utils.data import Dataset, DataLoader

PyTorch version: 2.9.1


In [ ]:
# Class to load dataset end GPT-2 encode
class GPTDatasetV1(Dataset):
  def __init__(self, txt, tokenizer, max_length, stride):
    self.input_ids = []
    self.target_ids = []

    # Tokenize the entire text
    token_ids = tokenizer.encode(txt, allowed_special={ "<|endoftext|>" })
    assert len(token_ids) > max_length, "Number of tokenized inputs must at least be equal to max_length+1"

    # Use a sliding window to chunk the book into overlapping sequences of max_length
    for i in range(0, len(token_ids) - max_length, stride):
      input_chunk = token_ids[i:i + max_length]
      target_chunk = token_ids[i + 1: i + max_length + 1]
      self.input_ids.append(torch.tensor(input_chunk))
      self.target_ids.append(torch.tensor(target_chunk))

  def __len__(self):
    return len(self.input_ids)

  def __getitem__(self, idx):
    return self.input_ids[idx], self.target_ids[idx]

In [ ]:
# Function to create encoding dataloader
def create_dataloader_v1(text, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):

  # Initialize the tokenizer
  tokenizer = tiktoken.get_encoding("gpt2")

  # Create dataset
  dataset = GPTDatasetV1(text, tokenizer, max_length, stride)

  # Create iterable dataloader to load batches of data:
  # - batch_size loads batch of data of specified size during each iteration
  # - shuffle=True changes the order of data indices to achieve better generalization
  # - drop_last=True drops the last batch if it is shorter than the batch_size to prevent loss spikes during training
  dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)
  return dataloader

In [ ]:
# Encode the data in batch of size 1
dataloader = create_dataloader_v1(raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)
data_iter = iter(dataloader)

# Print first batch
inputs, targets = next(data_iter)
print("batch 1")
print("inputs", inputs)
print("targets", targets)

batch 1
inputs tensor([[  40,  367, 2885, 1464]])
targets tensor([[ 367, 2885, 1464, 1807]])


In [40]:
# Print second batch
inputs, targets = next(data_iter)
print("batch 2")
print("inputs", inputs)
print("targets", targets)

batch 2
inputs tensor([[ 367, 2885, 1464, 1807]])
targets tensor([[2885, 1464, 1807, 3619]])


In [ ]:
# Create data loader for batch of size 8 of 4 tokens
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)
data_iter = iter(dataloader)

# Print first batch
inputs, targets = next(data_iter)
print("batch 1")
print("inputs", inputs)
print("targets", targets)

batch 1
inputs tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
targets tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


In [37]:
# Print second batch
inputs, targets = next(data_iter)
print("batch 2")
print("inputs", inputs)
print("targets", targets)

batch 2
inputs tensor([[  287,   262,  6001,   286],
        [  465, 13476,    11,   339],
        [  550,  5710,   465, 12036],
        [   11,  6405,   257,  5527],
        [27075,    11,   290,  4920],
        [ 2241,   287,   257,  4489],
        [   64,   319,   262, 34686],
        [41976,    13,   357, 10915]])
targets tensor([[  262,  6001,   286,   465],
        [13476,    11,   339,   550],
        [ 5710,   465, 12036,    11],
        [ 6405,   257,  5527, 27075],
        [   11,   290,  4920,  2241],
        [  287,   257,  4489,    64],
        [  319,   262, 34686, 41976],
        [   13,   357, 10915,   314]])


In [48]:
# Create an embedding layer
vocab_size = 6
output_dim = 3
torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

# Show the 6x3 weight matrix of small random values within the embedding layer
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


In [ ]:
# Demonstrate one-hot encoding vector applied to embedding layer
input_ids = torch.tensor([2, 3, 5, 1])
print(embedding_layer(input_ids))

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


In [50]:
# Create an embedding layer for GPT-3 tokenizer
vocab_size = 50257
output_dim = 256
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [51]:
# Create data loader for batch of size 8 of 4 tokens
max_length = 4
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=max_length, stride=max_length, shuffle=False)

# Print
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("batch 1")
print("inputs", inputs)
print("targets", targets)

batch 1
inputs tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
targets tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])
